In [1]:
import json
import time
from typing import Dict, Any, List, Optional, Tuple, Set, Callable, Union
from pathlib import Path

import requests

# ============================================================
# PIPELINE 2 (THREAD-BASED): THREAD-LEVEL-METRICS
#  - argument_novelty  (in [0,1])
#  - semantic_entropy  (>= 0, nicht nach oben begrenzt)
#
# Thread-Kontext: IMMER die direkte Parent-Kette nach oben,
#                 bis parent_id == link_id (bzw. submission_id) ODER Root (t3_/None).
#                 NUR der direkte Weg (keine Geschwister/Children).
#
# Input: NDJSON der ersten Pipeline (mit arguments, parent_id, submission_id/link_id, usw.)
# Output: wahlweise
#   A) pro (comment_id, thread_task) eine Zeile (klassischer Pipeline2-Output)
#   B) "enriched": gleiche Zeilenanzahl wie Input, aber mit thread_metrics angehängt
# ============================================================


# ------------------------
# Configuration
# ------------------------
LMSTUDIO_BASE_URL = "http://127.0.0.1:1234"  # Change if LM Studio runs on a different host/port
MODEL_NAME = "ibm/granite-3.2-8b"  # e.g., "qwen2.5-7b-instruct"
TIMEOUT_SECONDS = 60  # HTTP request timeout
MAX_RETRIES = 3  # Max attempts per (comment, task)
RETRY_BACKOFF_SECONDS = 1.5  # Exponential backoff base

# ------------------------
# Task-Spezifikationen (Pipeline 2)
# ------------------------
THREAD_TASK_SPECS: Dict[str, Dict[str, Any]] = {
    "argument_novelty": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "argument_novelty",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, 1.0),  # 0 = kein neues Argument, 1 = komplett neu
        "allow_abstain": True,
        "label_key": None,
    },
    "semantic_entropy": {
        "schema_keys": {"task", "score", "confidence"},
        "task_value": "semantic_entropy",
        "score_key": "score",
        "score_type": (int, float),
        "confidence_range": (0.0, 1.0),
        "score_range": (0.0, float("inf")),
        "allow_abstain": True,
        "label_key": None,
    },
}

# ------------------------
# System Prompt für Pipeline 2
# ------------------------
THREAD_SYSTEM_PROMPT = """You are an annotator estimating discourse-level metrics for a given comment
in the context of all previous comments in the discussion thread.

You receive:
{
  "CURRENT_ARGUMENTS": [ ... ],   // arguments extracted for the current comment
  "HISTORY_ARGUMENTS": [ ... ]    // arguments from ALL earlier comments in the thread,
                                  // following the parent_id chain back to the root
}

HISTORY_ARGUMENTS is ordered from earliest ancestor to the immediate parent.

Your tasks:

1) Argument novelty (argument_novelty)
--------------------------------------
TASK: Estimate how much of the CURRENT_ARGUMENTS content is *novel* relative to HISTORY_ARGUMENTS.

Score: float in [0,1].
- 0.0 = entirely re-uses arguments already present in HISTORY_ARGUMENTS
- 0.5 = mix of re-used and somewhat new arguments
- 1.0 = introduces entirely new arguments not present before

Heuristics:
- If HISTORY_ARGUMENTS is empty (no previous comments), default to a high score (e.g. 0.8-1.0),
  unless CURRENT_ARGUMENTS itself is empty or trivial.
- If CURRENT_ARGUMENTS is empty, return "ABSTAIN" (you cannot judge novelty).

JSON schema:
{
  "task": "argument_novelty",
  "score": <float in [0.0,1.0] OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

2) Semantic entropy (semantic_entropy)
--------------------------------------
TASK: Approximate the semantic diversity of arguments in the thread up to and including the current comment.

Intuition: a Shannon-entropy-like measure of the argument-topic distribution.
- Low entropy (~0): almost all arguments are about one narrow theme.
- Medium entropy (~1-2): several recurring argument themes.
- High entropy (>2): many different themes with relatively balanced presence.

Score: non-negative float (>= 0), no strict upper bound, but usual values should stay in a
reasonable range (e.g., 0 to 3).

Heuristics:
- Base your judgement on how many distinct argument themes you see in
  HISTORY_ARGUMENTS + CURRENT_ARGUMENTS, and how balanced they are.
- If there are no arguments at all, or almost no content, return "ABSTAIN".

JSON schema:
{
  "task": "semantic_entropy",
  "score": <float >= 0.0 OR "ABSTAIN">,
  "confidence": <float in [0,1]>
}

General rules:
--------------
- Prefer "ABSTAIN" if information is too weak, arguments are empty, or you cannot judge reliably.
- Output MUST be strictly valid JSON with the exact schema above.
"""

THREAD_TASK_INSTRUCTION_TEMPLATE = (
    "Now perform ONLY the task = {task_name} on the given CURRENT_ARGUMENTS and HISTORY_ARGUMENTS. "
    "Return strictly valid JSON for that task and nothing else (no Markdown). "
    "Ensure keys and value ranges match the schema exactly."
)


# ------------------------
# HTTP call utilities
# ------------------------
def call_lmstudio_chat(messages: List[Dict[str, str]], temperature: float = 0.0) -> str:
    """
    Call LM Studio (OpenAI-compatible) Chat Completions API and return raw text.
    Raises requests.RequestException on network/HTTP errors.
    """
    url = f"{LMSTUDIO_BASE_URL}/v1/chat/completions"
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": temperature,
        "stream": False,
    }
    resp = requests.post(url, json=payload, timeout=TIMEOUT_SECONDS)
    resp.raise_for_status()
    data = resp.json()
    return data["choices"][0]["message"]["content"].strip()


def _is_number(x: Any) -> bool:
    return isinstance(x, (int, float))


# ------------------------
# Validation für Pipeline 2
# ------------------------
def validate_thread_response(task: str, obj: Dict[str, Any]) -> Optional[str]:
    """Validate response for THREAD_TASK_SPECS (argument_novelty, semantic_entropy)."""
    spec = THREAD_TASK_SPECS[task]

    if not isinstance(obj, dict):
        return "Response is not a JSON object"

    missing = spec["schema_keys"] - set(obj.keys())
    if missing:
        return f"Missing required keys: {sorted(missing)}"

    if obj.get("task") != spec["task_value"]:
        return f'Field "task" must be "{spec["task_value"]}"'

    conf = obj.get("confidence")
    if not _is_number(conf):
        return '"confidence" must be a number'
    lo_c, hi_c = spec["confidence_range"]
    if not (lo_c <= conf <= hi_c):
        return f'"confidence" must be in [{lo_c}, {hi_c}]'

    score_key = spec["score_key"]
    val = obj.get(score_key)

    # ABSTAIN erlaubt?
    if isinstance(val, str):
        if spec.get("allow_abstain") and val == "ABSTAIN":
            return None
        return f'"{score_key}" must be a number or "ABSTAIN"'

    if not _is_number(val):
        return f'"{score_key}" must be a number'

    lo, hi = spec["score_range"]
    if not (lo <= float(val) <= hi):
        return f'"{score_key}" out of range [{lo}, {hi}]'

    return None


# ------------------------
# NDJSON writer
# ------------------------
def write_ndjson_line(fp, obj: Dict[str, Any]) -> None:
    fp.write(json.dumps(obj, ensure_ascii=False) + "\n")


# ------------------------
# Prompt für Pipeline 2 bauen
# ------------------------
def build_thread_user_prompt(
        task: str,
        current_arguments: List[str],
        history_arguments: List[str],
) -> str:
    payload = {
        "CURRENT_ARGUMENTS": current_arguments,
        "HISTORY_ARGUMENTS": history_arguments,
    }
    directive = THREAD_TASK_INSTRUCTION_TEMPLATE.format(task_name=task)
    return json.dumps(payload, ensure_ascii=False) + "\n\n" + directive


# ------------------------
# Ein Task-Aufruf (Pipeline 2)
# ------------------------
def annotate_thread_metric(
        task: str,
        current_arguments: List[str],
        history_arguments: List[str],
) -> Dict[str, Any]:
    """
    Run one thread-level task (argument_novelty or semantic_entropy) for a given comment,
    given its CURRENT_ARGUMENTS and HISTORY_ARGUMENTS (from ancestor comments).
    """
    last_err = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            messages = [
                {"role": "system", "content": THREAD_SYSTEM_PROMPT},
                {"role": "user", "content": build_thread_user_prompt(task, current_arguments, history_arguments)},
            ]
            raw = call_lmstudio_chat(messages, temperature=0.0)
            obj = json.loads(raw)

            err = validate_thread_response(task, obj)
            if err is None:
                return obj
            last_err = f"Thread schema validation failed (attempt {attempt}): {err}"

        except requests.RequestException as e:
            last_err = f"HTTP error (attempt {attempt}): {e}"
        except json.JSONDecodeError as e:
            last_err = f"JSON parse error (attempt {attempt}): {e}"
        except Exception as e:
            last_err = f"Unexpected error (attempt {attempt}): {e}"

        time.sleep((RETRY_BACKOFF_SECONDS ** attempt))

    return {
        "task": task,
        "error": last_err or "Unknown error",
    }


# ============================================================
# Thread / ID Utilities
# ============================================================
def _to_str_or_none(x: Any) -> Optional[str]:
    if x is None:
        return None
    if isinstance(x, float) and x != x:  # NaN
        return None
    s = str(x).strip()
    return s if s else None


def _split_fullname(x: Any) -> Tuple[Optional[str], Optional[str]]:
    """
    Reddit-typische Fullnames:
      t1_<comment_id>
      t3_<submission_id>
    Rückgabe: (kind, id_without_prefix)
      kind in {"t1","t3",None}
    """
    s = _to_str_or_none(x)
    if s is None:
        return (None, None)
    if s.startswith("t1_"):
        return ("t1", s[3:])
    if s.startswith("t3_"):
        return ("t3", s[3:])
    return (None, s)


def _canonical_id(x: Any) -> Optional[str]:
    """Strip t1_/t3_ if present; return bare ID or None."""
    _, bare = _split_fullname(x)
    return bare


def _pick_link_id_from_record(obj: Dict[str, Any]) -> Optional[str]:
    """
    Best-effort: link_id / submission_id kann an unterschiedlichen Stellen liegen.
    Priorität:
      1) obj["link_id"]
      2) obj["submission_id"]
      3) obj["meta"]["submission_id"]
      4) obj["meta"]["thread_id"]
    Rückgabe immer canonical (ohne t3_/t1_).
    """
    for k in ("link_id", "submission_id"):
        if k in obj and _to_str_or_none(obj.get(k)) is not None:
            return _canonical_id(obj.get(k))
    meta = obj.get("meta")
    if isinstance(meta, dict):
        for k in ("submission_id", "thread_id", "link_id"):
            if k in meta and _to_str_or_none(meta.get(k)) is not None:
                return _canonical_id(meta.get(k))
    return None


# ============================================================
# Loader: Input NDJSON (Pipeline 1 Output)
#  - preserves ALL fields (dimensions)
#  - builds:
#     - per-record list (to optionally enrich output 1:1)
#     - per-comment map (for parent chain + arguments)
# ============================================================
def load_records_and_comment_map_from_labels_ndjson(
        ndjson_path: Union[str, Path],
        *,
        prefer_nonempty_arguments: bool = True,
) -> Tuple[List[Dict[str, Any]], Dict[str, Dict[str, Any]]]:
    """
    Returns:
      records: List[dict]  (alle Zeilen aus Input NDJSON, unverändert)
      comment_map: Dict[canonical_comment_id] -> {
          "comment_id": <canonical>,
          "comment_id_raw": <as seen>,
          "parent_id_raw": ...,
          "parent_id_canon": ...,
          "link_id_canon": ...,
          "arguments": [...],
          "arguments_confidence": ...,
          "base_fields": { ... alle originalen keys, aber task/result als source_* umbenannt ... }
      }

    Hinweis:
      Pipeline 1 hat typischerweise mehrere Zeilen pro comment_id (unterschiedliche Tasks),
      aber "arguments" sind pro comment_id identisch. Wir nehmen pro comment_id die "beste"
      Zeile als base_fields (heuristisch: mehr arguments gewinnt; sonst erste).
    """
    ndjson_path = Path(ndjson_path)
    records: List[Dict[str, Any]] = []

    # collect candidates per comment
    candidates: Dict[str, List[Dict[str, Any]]] = {}

    with ndjson_path.open("r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue

            records.append(obj)

            cid_raw = obj.get("comment_id", obj.get("id"))
            cid_canon = _canonical_id(cid_raw)
            if cid_canon is None:
                continue

            candidates.setdefault(cid_canon, []).append(obj)

    comment_map: Dict[str, Dict[str, Any]] = {}

    def _quality(o: Dict[str, Any]) -> Tuple[int, int, int]:
        """
        Höher ist besser:
          - Anzahl arguments
          - hat body?
          - hat created_utc?
        """
        args = o.get("arguments", [])
        n_args = len(args) if isinstance(args, list) else 0
        has_body = 1 if _to_str_or_none(o.get("body")) is not None else 0
        has_time = 1 if _to_str_or_none(o.get("created_utc")) is not None else 0
        return (n_args, has_body, has_time)

    for cid_canon, objs in candidates.items():
        if not objs:
            continue

        # choose best base object
        if prefer_nonempty_arguments:
            objs_sorted = sorted(objs, key=_quality, reverse=True)
            base_obj = objs_sorted[0]
        else:
            base_obj = objs[0]

        # extract ids
        parent_raw = base_obj.get("parent_id", base_obj.get("parent"))
        parent_kind, parent_bare = _split_fullname(parent_raw)

        link_id_canon = _pick_link_id_from_record(base_obj)

        # canonical parent: strip t1_ / t3_ but keep None
        parent_canon = parent_bare

        # preserve ALL dimensions: keep base fields, but avoid collision with Pipeline2 task/result
        base_fields = dict(base_obj)  # shallow copy
        if "task" in base_fields:
            base_fields["source_task"] = base_fields.pop("task")
        if "result" in base_fields:
            base_fields["source_result"] = base_fields.pop("result")

        comment_map[cid_canon] = {
            "comment_id": cid_canon,
            "comment_id_raw": base_obj.get("comment_id", base_obj.get("id")),
            "parent_id_raw": parent_raw,
            "parent_kind": parent_kind,  # None / t1 / t3
            "parent_id_canon": parent_canon,  # bare id
            "link_id_canon": link_id_canon,  # bare id (submission/link)
            "arguments": base_obj.get("arguments", []) if isinstance(base_obj.get("arguments", []), list) else [],
            "arguments_confidence": base_obj.get("arguments_confidence", None),
            "base_fields": base_fields,  # includes ALL original dimensions (with source_task/source_result)
        }

    return records, comment_map


# ============================================================
# Thread-context builder (DIRECT ancestor chain only)
#  - stop at:
#       * parent is None
#       * parent_kind == "t3"  (submission root)
#       * parent_id_canon == link_id_canon (desired stopping rule)
#       * parent_id_canon not in comment_map (broken chain)
# ============================================================
def build_history_arguments_for_comment(
        comment_id_canon: str,
        comment_map: Dict[str, Dict[str, Any]],
) -> List[str]:
    """
    Baut HISTORY_ARGUMENTS für einen Kommentar auf:
      - Iterativ über parent_id entlang der direkten Parent-Kette
      - Stop bis parent_id == link_id (oder t3_ / None)
      - Reihenfolge: frühester Vorfahre zuerst, unmittelbarer Parent zuletzt.
    """
    if comment_id_canon not in comment_map:
        return []

    entry = comment_map[comment_id_canon]
    link_id = entry.get("link_id_canon")
    current_parent = entry.get("parent_id_canon")
    current_parent_kind = entry.get("parent_kind")  # kind of *this* comment's parent raw, first hop only

    history_pairs: List[Tuple[str, List[str]]] = []
    seen: Set[str] = set()

    # Walk up parents (direct path only)
    # Note: For subsequent hops, we use parent's stored parent_kind derived from their parent raw.
    while True:
        if current_parent is None:
            break
        if link_id is not None and current_parent == link_id:
            break

        # If the FIRST hop parent was a t3_ fullname, that already implies root
        if current_parent_kind == "t3":
            break

        if current_parent in seen:
            break
        if current_parent not in comment_map:
            break

        seen.add(current_parent)
        parent_entry = comment_map[current_parent]
        parent_args = parent_entry.get("arguments", [])
        history_pairs.append((current_parent, parent_args if isinstance(parent_args, list) else []))

        # move up
        current_parent = parent_entry.get("parent_id_canon")
        current_parent_kind = parent_entry.get("parent_kind")

    # root -> parent order
    history_pairs.reverse()

    # flatten with cid tags (as in your original code)
    history_arguments: List[str] = []
    for cid, args in history_pairs:
        for a in args:
            history_arguments.append(f"[{cid}] {a}")

    return history_arguments


# ============================================================
# MAIN: Pipeline 2 (Option A) - per (comment_id, thread_task) output
#   Output schema per line:
#   {
#     ... ALL original dimensions from base record (task/result renamed to source_*) ...
#     "task": "argument_novelty" | "semantic_entropy",
#     "result": { ...thread-task-json... },
#     "current_arguments": [...],
#     "history_arguments": [...]
#   }
# ============================================================
def run_thread_metrics_pipeline(
        ndjson_in: Union[str, Path],
        ndjson_out: Union[str, Path] = "thread_metrics.ndjson",
        tasks: Optional[List[str]] = None,
) -> Dict[str, Any]:
    if tasks is None:
        tasks = list(THREAD_TASK_SPECS.keys())

    _, comment_map = load_records_and_comment_map_from_labels_ndjson(ndjson_in)

    comment_ids = list(comment_map.keys())
    written_records = 0
    error_count = 0

    ndjson_out = Path(ndjson_out)
    ndjson_out.parent.mkdir(parents=True, exist_ok=True)

    with ndjson_out.open("w", encoding="utf-8") as fp:
        for cid in comment_ids:
            entry = comment_map[cid]

            current_args = entry.get("arguments", [])
            history_args = build_history_arguments_for_comment(cid, comment_map)

            for task in tasks:
                try:
                    thread_result = annotate_thread_metric(
                        task=task,
                        current_arguments=current_args,
                        history_arguments=history_args,
                    )

                    out = dict(entry.get("base_fields", {}))  # preserves all original dimensions
                    out.update({
                        # Pipeline2 outputs
                        "task": task,
                        "result": thread_result,
                        "current_arguments": current_args,
                        "history_arguments": history_args,
                        # helpful explicit ids (canonical)
                        "comment_id_canon": cid,
                        "parent_id_canon": entry.get("parent_id_canon"),
                        "link_id_canon": entry.get("link_id_canon"),
                    })

                    write_ndjson_line(fp, out)
                    written_records += 1
                except Exception:
                    error_count += 1

    return {
        "num_comments": len(comment_ids),
        "written_records": written_records,
        "errors": error_count,
        "ndjson_out": str(ndjson_out),
    }


# ============================================================
# OPTIONAL: Pipeline 2 (Option B) - ENRICH original NDJSON 1:1
#   Output has SAME number of lines as input; each line gets:
#     "thread_metrics": {
#         "argument_novelty": { ... },
#         "semantic_entropy": { ... }
#     }
#     "thread_history_arguments": [...]
#
#   This preserves the "task dimension" from Pipeline 1 completely (no collapsing).
# ============================================================
def run_thread_metrics_pipeline_enrich(
        ndjson_in: Union[str, Path],
        ndjson_out: Union[str, Path] = "labels_enriched_with_thread_metrics.ndjson",
        tasks: Optional[List[str]] = None,
) -> Dict[str, Any]:
    if tasks is None:
        tasks = list(THREAD_TASK_SPECS.keys())

    records, comment_map = load_records_and_comment_map_from_labels_ndjson(ndjson_in)

    # compute thread metrics once per comment_id
    per_comment_metrics: Dict[str, Dict[str, Any]] = {}
    per_comment_history: Dict[str, List[str]] = {}

    comment_ids = list(comment_map.keys())
    for cid in comment_ids:
        entry = comment_map[cid]
        current_args = entry.get("arguments", [])
        history_args = build_history_arguments_for_comment(cid, comment_map)

        per_comment_history[cid] = history_args
        per_comment_metrics[cid] = {}

        for task in tasks:
            per_comment_metrics[cid][task] = annotate_thread_metric(
                task=task,
                current_arguments=current_args,
                history_arguments=history_args,
            )

    ndjson_out = Path(ndjson_out)
    ndjson_out.parent.mkdir(parents=True, exist_ok=True)

    written_records = 0
    error_count = 0

    with ndjson_out.open("w", encoding="utf-8") as fp:
        for obj in records:
            cid_canon = _canonical_id(obj.get("comment_id", obj.get("id")))
            if cid_canon is None or cid_canon not in per_comment_metrics:
                # write unchanged if we can't map it
                write_ndjson_line(fp, obj)
                written_records += 1
                continue

            enriched = dict(obj)
            enriched["thread_history_arguments"] = per_comment_history.get(cid_canon, [])
            enriched["thread_metrics"] = per_comment_metrics.get(cid_canon, {})
            enriched["comment_id_canon"] = cid_canon
            enriched["parent_id_canon"] = comment_map[cid_canon].get("parent_id_canon")
            enriched["link_id_canon"] = comment_map[cid_canon].get("link_id_canon")

            write_ndjson_line(fp, enriched)
            written_records += 1

    return {
        "input_records": len(records),
        "written_records": written_records,
        "errors": error_count,
        "ndjson_out": str(ndjson_out),
    }


# ============================================================
# Example usage
# ============================================================
if __name__ == "__main__":
    # A) klassisch: pro (comment_id, thread_task) neue Zeilen
    stats_a = run_thread_metrics_pipeline(
        ndjson_in="labels_3000.ndjson",
        ndjson_out="thread_metrics.ndjson",
        tasks=["argument_novelty", "semantic_entropy"],
    )
    print("Pipeline A:", stats_a)

    # B) enriched: gleiche Zeilenanzahl wie Input, nur erweitert
    stats_b = run_thread_metrics_pipeline_enrich(
        ndjson_in="labels_3000.ndjson",
        ndjson_out="labels_3000_enriched.ndjson",
        tasks=["argument_novelty", "semantic_entropy"],
    )
    print("Pipeline B:", stats_b)


Pipeline A: {'num_comments': 578, 'written_records': 1156, 'errors': 0, 'ndjson_out': 'thread_metrics.ndjson'}
Pipeline B: {'input_records': 4041, 'written_records': 4041, 'errors': 0, 'ndjson_out': 'labels_3000_enriched.ndjson'}
